# Kaggle launcher: transfer and external evaluation
Stage 0 artifact. Enable a Kaggle GPU and attach the recorded XLM-R checkpoints. This uses the completed external-notebook lineage, not the lowercase duplicate whose stored output contains an unpickling error.

In [ ]:
from pathlib import Path
import shutil, subprocess, sys

RUN_HEAVY = False
SOURCE_NOTEBOOKS = [
    Path('Phase 6 (Architectures)/Transfer Learning/transfer-learning.ipynb'),
    Path('External Hate Dataset Testing/External hate Dataset test.ipynb'),
]
REQUIRED_INPUTS = [
    Path('/kaggle/input/datasets/iftekharuddin27/preprocessed-datasets'),
    Path('/kaggle/input/datasets/iftekharuddin27/transformer-learning'),
    Path('/kaggle/input/datasets/faisalshanto/external-hate-dataset'),
]

def find_repo():
    candidates = [Path('/kaggle/working/Capstone-Project')]
    candidates.extend(Path('/kaggle/input').glob('*/Capstone-Project'))
    for candidate in candidates:
        if (candidate / 'README.md').is_file():
            return candidate
    raise FileNotFoundError('Attach the repository as a Kaggle dataset or place it at /kaggle/working/Capstone-Project')

def execute_copy(repo_root, relative_path):
    source = repo_root / relative_path
    if not source.is_file():
        raise FileNotFoundError(source)
    target = Path('/kaggle/working') / ('executed_' + source.name)
    shutil.copy2(source, target)
    subprocess.run([sys.executable, '-m', 'jupyter', 'nbconvert', '--to', 'notebook', '--execute', '--ExecutePreprocessor.timeout=-1', '--output', target.name, str(target)], check=True, cwd=target.parent)

if RUN_HEAVY:
    if not __import__('torch').cuda.is_available():
        raise RuntimeError('Enable a Kaggle GPU before transfer or external inference.')
    missing = [str(path) for path in REQUIRED_INPUTS if not path.exists()]
    if missing:
        raise FileNotFoundError('Missing Kaggle inputs: ' + ', '.join(missing))
    root = find_repo()
    for notebook in SOURCE_NOTEBOOKS:
        execute_copy(root, notebook)
else:
    print('Stage 0 guard active: no experiment executed. Review paths, then set RUN_HEAVY=True on Kaggle.')